Practical No. 05 - Optimization of genetic algorithm parameter in hybrid
genetic algorithm-neural network
modelling: Application to spray drying of coconut milk.

In [ ]:
!pip install deap
!pip install pandas
!pip install tensorflow
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from deap import base, creator, tools, algorithms
import random
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Defaulting to user installation because normal site-packages is not writeable
  Using cached deap-1.4.4-py3-none-any.whl.metadata (13 kB)
  Using cached moocore-0.3.1-cp310-abi3-win_amd64.whl.metadata (6.7 kB)
Using cached deap-1.4.4-py3-none-any.whl (93 kB)
Using cached moocore-0.3.1-cp310-abi3-win_amd64.whl (517 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 3.0 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/12.6 MB 2.8 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.6 MB 2.9 MB/s eta 0:00:04
   -------- ------------------------------- 2.6/12.6 MB 2.8 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/12.6 MB 3.0 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.6 MB 2.9 MB/s eta 0:00:03
   -------------- ------------------------- 4.5/12.6 MB 2.8 MB/s eta 0:00:03
 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 8.5 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.9 MB 12.2 MB/s eta 0:00:01
   -------------- ------------------------- 3.7/9.9 MB 9.1 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.9 MB 6.7 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.9 MB 5.8 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.9 MB 5.5 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.9 MB 5.1 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 4.8 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/9.9 MB 4.5 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/9.9 MB 4.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.9 MB 4.3 MB/s eta 0:00:01
   -------------

In [ ]:
# Simulate dataset (replace with real-world data if available)
np.random.seed(42)
n_samples = 100
X = np.random.uniform(low=30, high=100, size=(n_samples, 2)) # e.g.,temperature, feed rate
y = 0.8 * X[:, 0] - 0.5 * X[:, 1] + np.random.normal(0, 5, n_samples) #Spray drying efficiency
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)
# Normalize inputs
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

In [ ]:
def build_nn(num_neurons, learning_rate):
    model = Sequential([
        Dense(num_neurons, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(1)
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse'
    )
    
    return model

In [ ]:

def evaluate_fitness(individual):
    num_neurons = int(individual[0])  # Number of neurons
    learning_rate = individual[1]     # Learning rate

    model = build_nn(num_neurons, learning_rate)
    model.fit(X_train, y_train, epochs=20, verbose=0, batch_size=10)

    # Predict and calculate MSE
    y_pred = model.predict(X_test, verbose=0)
    mse = mean_squared_error(y_test, y_pred)

    return (mse,)   # Must return tuple


# -------- GA CONFIGURATION --------

toolbox = base.Toolbox()

# Avoid redefining if already created (important in Jupyter)
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize MSE

if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

# Attributes
toolbox.register("attr_num_neurons", random.randint, 5, 50)
toolbox.register("attr_learning_rate", random.uniform, 0.001, 0.01)

# Individual & Population
toolbox.register(
    "individual",
    tools.initCycle,
    creator.Individual,
    (toolbox.attr_num_neurons, toolbox.attr_learning_rate),
    n=1
)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# GA Operators
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_fitness)


# -------- RUN GA --------

random.seed(42)

population = toolbox.population(n=10)

ngen = 20
cxpb = 0.5
mutpb = 0.2

result, log = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=cxpb,
    mutpb=mutpb,
    ngen=ngen,
    verbose=True
)

In [ ]:
best_individual = tools.selBest(population, k=1)[0]
print(f"Best Individual: Neurons={best_individual[0]}, Learning Rate={best_individual[1]}")
# Train and evaluate the final model
final_model = build_nn(int(best_individual[0]), best_individual[1])
final_model.fit(X_train, y_train, epochs=50, verbose=1, batch_size=10)

In [ ]:
# Test the model
y_pred = final_model.predict(X_test)
final_mse = mean_squared_error(y_test, y_pred)
print(f"Final MSE on Test Data: {final_mse}")

In [ ]:
# Plot true vs predicted values
plt.scatter(y_test, y_pred, c='blue', label='Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)],
color='red', linestyle='--')
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.title("True vs Predicted Values")
plt.show()